# S1 · Arquitectura Big Data y puesta en marcha

AeroStreamPeru · Equipo 04 · Ciclo 9

Este laboratorio verifica que PySpark puede leer un bloque del histórico Bronze de OpenSky. El brief fija el alcance y el significado de los indicadores.

## Arquitectura Lambda

- **Batch (U1)** · Trino → Parquet Bronze → PySpark → resultados históricos y modelos.
- **Velocidad (U2)** · API de OpenSky → Kafka → Spark Structured Streaming → Grafana.

El histórico permite recalcular indicadores sobre períodos completos. La capa en tiempo real está prevista para actualizar las lecturas y las predicciones a medida que llegan capturas.

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master('local[2]')
    .appName('AeroStreamPeru-S01')
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)
print(f'Spark {spark.version}')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 14:57:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0


In [2]:
bronze = Path('/opt/s01-arquitectura/data/2026')
bloque = bronze / '01' / '2026-01-01_2026-01-05.parquet'

estados = spark.read.parquet(str(bloque))
estados.printSchema()
print(f'Filas: {estados.count():,} · Columnas: {len(estados.columns)}')
estados.select('time', 'icao24', 'onground', 'velocity').show(5)

root
 |-- time: long (nullable = true)
 |-- icao24: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- velocity: double (nullable = true)
 |-- heading: double (nullable = true)
 |-- vertrate: double (nullable = true)
 |-- callsign: string (nullable = true)
 |-- onground: boolean (nullable = true)
 |-- alert: boolean (nullable = true)
 |-- spi: boolean (nullable = true)
 |-- squawk: string (nullable = true)
 |-- baroaltitude: double (nullable = true)
 |-- geoaltitude: double (nullable = true)
 |-- lastposupdate: double (nullable = true)
 |-- lastcontact: double (nullable = true)
 |-- hour: long (nullable = true)

Filas: 2,365,631 · Columnas: 17
+----------+------+--------+------------------+
|      time|icao24|onground|          velocity|
+----------+------+--------+------------------+
|1767466277|e8047a|   false|229.59941756235077|
|1767466278|e8047a|   false|229.59941756235077|
|1767466279|e8047a|   false|229.59941756235077|
|1767466280

In [3]:
spark.stop()